## Week 01/02 Homework Submission

This is the homwork submission for Week 1 & 2 of AI Engineering Buildcamp by Alexey Grigorev.

## Imports and Data Download

In [89]:
from openai import OpenAI
openai_client = OpenAI()

In [88]:
import pandas as pd 
pd.read_csv('books.csv')

,title,book_url,pdf_url
0,Think Python 2e,https://greenteapress.com/wp/think-python-2e/,http://greenteapress.com/thinkpython2/thinkpyt...
1,Think DSP,https://greenteapress.com/wp/think-dsp/,http://greenteapress.com/thinkdsp/thinkdsp.pdf
2,Think Complexity 2e,https://greenteapress.com/wp/think-complexity/,http://greenteapress.com/complexity2/thinkcomp...
3,Think Java 2e,https://greenteapress.com/wp/think-java-2e/,http://greenteapress.com/thinkjava7/thinkjava2...
4,Physical Modeling in MATLAB,https://greenteapress.com/wp/physical-modeling...,https://github.com/AllenDowney/PhysicalModelin...
5,Think OS,https://greenteapress.com/wp/think-os/,http://greenteapress.com/thinkos/thinkos.pdf
6,Think C++,https://greenteapress.com/wp/think-c/,https://raw.githubusercontent.com/tscheffl/Thi...


Write a script to download all the PDFs from the CSV file. You can save them anywhere you want. For example, the books/ directory.

In [ ]:
import requests
from pathlib import Path

Path("books").mkdir(exist_ok=True)

df = pd.read_csv("books.csv")

for _, row in df.iterrows():
    url = row["pdf_url"]
    filename = url.split("/")[-1]
    dest = Path("books") / filename
    
    if dest.exists():
        print(f"Skipping {filename} (already exists)")
        continue
    
    print(f"Downloading {filename}...")
    response = requests.get(url)
    response.raise_for_status()
    dest.write_bytes(response.content)
    print(f"Saved {filename}")


Skipping thinkpython2.pdf (already exists)
Saved thinkdsp.pdf
Saved thinkcomplexity2.pdf
Saved thinkjava2.pdf
Saved PhysicalModelingInMatlab4.pdf
Saved thinkos.pdf
Saved Think-C.pdf


### Q1. Converting PDFs to Markdowns

* The markitdown[pdf] library can convert various document formats including PDF to markdown or text format.

* Convert all the downloaded PDFs to markdown files and save them to a books_text/ directory.

In [ ]:
from markitdown import MarkItDown

md = MarkItDown(enable_plugins=False)
Path("books_text").mkdir(exist_ok=True)

for pdf_path in Path("books").glob("*.pdf"):
    md_path = Path("books_text") / pdf_path.with_suffix(".md").name
    
    if md_path.exists():
        print(f"Skipping {pdf_path.name} (already converted)")
        continue
    
    print(f"Converting {pdf_path.name}...")
    result = md.convert(str(pdf_path))
    md_path.write_text(result.text_content)
    print(f"Saved {md_path.name}")


Converting thinkcomplexity2.pdf...
Saved thinkcomplexity2.md
Converting thinkpython2.pdf...
Saved thinkpython2.md
Converting PhysicalModelingInMatlab4.pdf...
Saved PhysicalModelingInMatlab4.md
Converting thinkjava2.pdf...
Saved thinkjava2.md
Converting thinkdsp.pdf...
Saved thinkdsp.md
Converting thinkos.pdf...
Saved thinkos.md
Converting Think-C.pdf...
Saved Think-C.md


* How many lines are in the extracted content from the "Think Python" book?

In [109]:
%%bash
wc -l books_text/thinkpython2.md

   16268 books_text/thinkpython2.md


### Q2. Chunking for RAG

For RAG we need to split documents into smaller chunks.

First, prepare your documents:
1. Read each markdown file from your books_text/ directory
2. Split the content into lines
3. Remove empty lines and lines that contain only whitespace
4. Turn each book into a dictionary with source (filename) and a content (list of non-empty lines)

After that, chunk it.

In [ ]:
from gitsource import chunk_documents

books = []

for md_path in Path("books_text").glob("*.md"):
    print(f"Processing {md_path.name}...")
    text = md_path.read_text()
    lines = text.splitlines()
    non_empty_lines = [line.strip() for line in lines if line.strip()]
    books.append({"source": md_path.stem, "content": non_empty_lines})

doc_chunks = chunk_documents(books, size=100, step=50)

Processing Think-C.md...
Processing PhysicalModelingInMatlab4.md...
Processing thinkos.md...
Processing thinkcomplexity2.md...
Processing thinkdsp.md...
Processing thinkjava2.md...
Processing thinkpython2.md...


The chunk_documents function uses a sliding window approach with these parameters:
* size=100: number of items per chunk
* step=50: how many items to move forward for each chunk

With size=100 and step=50, each chunk is about 4,400 characters or 780 words on average.

How many chunks are produced for the "Think Python" book with these settings?

In [53]:
len([chunk for chunk in doc_chunks if chunk["source"]=="thinkpython2"])

214

### Q3. Indexing with minsearch

Now we need to index our chunks so we can search through them.

Load all your chunked documents and create an index:

In [81]:
from minsearch import Index

def prepare_documents(chunks):
    prepared = []
    for chunk in chunks:
        content = "\n".join(chunk["content"])
        prepared.append({"source": chunk["source"], "content": content})
    return prepared

documents = prepare_documents(doc_chunks)
# # here you need to turn the lists into strings
# # e.g. with content = "\n".join(chunk["content"])

index = Index(
    text_fields=["content"],
    keyword_fields=["source"]
)

index.fit(documents)

How many documents (chunks) did you index?

In [82]:
len(index.docs)

1009

### Q4. Searching and RAG

Now let's search our index.

In [83]:
results = index.search("python function definition", num_results=5)

Look at the top result. Which book did it come from?

In [85]:
results[0]['source']

'thinkpython2'

### Q5. Full RAG

In [93]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer


Do RAG for "python function definition". What's the response?

In [90]:
rag("python function definition")

'In Python, a function is defined using the `def` keyword, followed by the function name and parentheses that may include parameters. The function body is indented below the definition line and contains the statements that perform the function\'s operation.\n\nHere’s a simple example of a function definition in Python:\n\n```python\ndef greet(name):\n    print("Hello, " + name + "!")\n```\n\nIn this example:\n- `def` is the keyword used to define a function.\n- `greet` is the name of the function.\n- `name` is a parameter that the function uses.\n- The `print` statement inside the function body executes when the function is called.\n\nTo call the function, you would use:\n\n```python\ngreet("Alice")\n```\n\nThis would output: `Hello, Alice!`\n\nFunctions can also return values using the `return` statement:\n\n```python\ndef add(a, b):\n    return a + b\n```\n\nYou would call this function like so:\n\n```python\nresult = add(5, 3)\nprint(result)  # Output: 8\n```\n\nThis structure allow

Now let's change these functions to also return the number of input and output tokens.

In [ ]:
def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return {
        "text": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens,
    }

How many input tokens did we use for this one RAG query?

In [97]:
answer = rag("python function definition")

In [98]:
print(answer['input_tokens'])

7262


### Q6. Structured vs Unstructured Output

In [102]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")


Modify the llm and rag functions from Question 5 to use structured outputs.
* do RAG for "python function definition"
* look at the number of input tokens
* compare the number with the results from Q5

In [105]:
def llm_structured(user_prompt,
                   output_type,
                   instructions=instructions,
                   model='gpt-4o-mini'):
    
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return {
        "text": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens,
    }

def rag_structured(query, output_type):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm_structured(user_prompt=prompt,
                            output_type=output_type,
                            instructions=instructions)
    return answer

In [106]:
answer_structured = rag_structured("python function definition", RAGResponse)

In [107]:
print(answer_structured['input_tokens'])

7448


In [108]:
print(answer_structured['input_tokens'] - answer['input_tokens'])

186
